In [3]:
import glob
import os
import numpy as np

import pandas as pd
import geopandas as gpd
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
from matplotlib.colors import ListedColormap
from GRSdriver import driver_S2_SAFE as S2

import json
import requests
from datetime import datetime
from multiprocessing import Pool
import subprocess

import GRSdriver

opj = os.path.join

GRSdriver.__version__

'1.0.7'

In [4]:
odir = '/data/satellite/Sentinel-2/L1C/'
date_start='2021-06-01T00:00:00.000Z' 
date_end ='2021-10-31T00:00:00.000Z'
tiles=['32TLR']

date_start='2017-01-01T00:00:00.000Z' 
date_end ='2020-12-31T00:00:00.000Z'
#tiles=['20PNC']#'10SEJ''31TFJ']#'31TFJ']['55WFU']
tiles = ['35TNH']
cloud_cover_max = 60      


cmd = []
for tile in tiles:
    json = requests.get("https://catalogue.dataspace.copernicus.eu/odata/v1/Products?$filter=Collection/Name eq 'SENTINEL-2' "+
                    " and Attributes/OData.CSC.DoubleAttribute/any(att:att/Name eq 'cloudCover' and att/OData.CSC.DoubleAttribute/Value lt {:.2f})".format(cloud_cover_max)+
                    " and Attributes/OData.CSC.StringAttribute/any(att:att/Name eq 'productType' and att/OData.CSC.StringAttribute/Value eq 'S2MSI1C')"+
                    " and contains(Name,'"+tile+"') "+
                    # next line is to keep latest baseline only (comment to get baseline < 4)
                    #" and (contains(Name,'_N04') or contains(Name,'_N05')) "+
                     #   " and (contains(Name,'_N05')) "+
                    " and ContentDate/Start gt "+date_start+" and ContentDate/Start lt "+date_end+"&$orderby=ContentDate/Start&$top=1000").json()
    df = pd.DataFrame.from_dict(json['value'])
    
    
    
    for irow,info in df.iterrows():
        #print(info['Name'])
        # get only rep N0500 to avoid duplicates (could be revised in the future)
        if info.S3Path.split('/')[4] == 'L1C_N0500':
        #if True:
            date=datetime.strptime(info['Name'].split('_')[2],'%Y%m%dT%H%M%S')
            date.year
            odir_ = opj(odir,tile,str(date.year),str(date.month).zfill(2),str(date.day).zfill(2))
            cmd.append('s3cmd -c ~/.s3cfg sync s3:/'+info['S3Path']+' '+odir_)
cmd

['s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/03/S2A_MSIL1C_20170103T090342_N0500_R007_T35TNH_20230919T002423.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/01/03',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/13/S2A_MSIL1C_20170113T090321_N0500_R007_T35TNH_20231005T132517.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/01/13',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/13/S2A_MSIL1C_20170113T090321_N0500_R007_T35TNH_20231005T035047.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/01/13',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/26/S2A_MSIL1C_20170126T091231_N0500_R050_T35TNH_20231025T181024.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/01/26',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/02/02/S2A_MSIL1C_20170202T090201_N0500_R007_T35TNH_20230925T012148.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/02/02',
 's3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C

In [5]:

def call(command):
    print(command)
    pipeline_out = subprocess.call(command, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT, shell=True)
    return

ncore=4
with Pool(processes=ncore) as pool:
    pool.map(call, cmd)
    pool.close

s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/01/03/S2A_MSIL1C_20170103T090342_N0500_R007_T35TNH_20230919T002423.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/01/03s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/07/25/S2A_MSIL1C_20170725T090601_N0500_R050_T35TNH_20231006T111611.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/07/25s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/05/13/S2A_MSIL1C_20170513T090021_N0500_R007_T35TNH_20231110T202640.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/05/13s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/10/10/S2A_MSIL1C_20171010T085901_N0500_R007_T35TNH_20231015T050459.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/10/10



s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/07/30/S2B_MSIL1C_20170730T091029_N0500_R050_T35TNH_20230912T011931.SAFE /data/satellite/Sentinel-2/L1C/35TNH/2017/07/30
s3cmd -c ~/.s3cfg sync s3://eodata/Sentinel-2/MSI/L1C_N0500/2017/05/18/S2B_

In [16]:
import requests

url = f"https://zipper.dataspace.copernicus.eu/odata/v1/Products("+df.Id[0]+")/$value"

headers = {"Authorization": f"Bearer {access_token}"}

session = requests.Session()
session.headers.update(headers)
response = session.get(url, headers=headers, stream=True)

with open("product.zip", "wb") as file:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            file.write(chunk)